# K-Means Algorithm Lab

K-means is an unsupervised learning method for clustering data points. The algorithm iteratively divides data points into K clusters by minimizing the variance in each cluster.

This notebook walks through:
1. Using `sklearn`'s `KMeans` with the elbow method to pick K
2. Building K-means **from scratch** (random init)
3. Seeing why random initialization can fail
4. Fixing it with **k-means++** initialization

Run cells top to bottom — the first cell installs any missing packages.

## Setup
Install dependencies (safe to run even if already installed).

In [ ]:
import sys
!{sys.executable} -m pip install -q numpy matplotlib scikit-learn seaborn

## Part 1: Quick Start with scikit-learn's KMeans

First we create a small toy dataset of (x, y) points.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans as SKLearnKMeans

x = [4, 5, 10, 4, 3, 11, 14, 6, 10, 12]
y = [21, 19, 24, 17, 16, 25, 24, 22, 21, 21]

data = list(zip(x, y))
print(data)

### Elbow method: choosing the best K

We only have 10 data points, so the maximum number of clusters is 10. For each value K in range(1,11), we train a K-means model and plot the inertia at that number of clusters.

In [ ]:
inertias = []

for i in range(1, 11):
    kmeans = SKLearnKMeans(n_clusters=i, n_init=10, random_state=42)
    kmeans.fit(data)
    inertias.append(kmeans.inertia_)

plt.plot(range(1, 11), inertias, marker='o')
plt.title('Elbow method')
plt.xlabel('Number of clusters')
plt.ylabel('Inertia')
plt.show()

We can see that the "elbow" on the graph above (where the inertia becomes more linear) is around K=2. We fit K-means one more time with K=2 and plot the resulting clusters.

In [ ]:
kmeans = SKLearnKMeans(n_clusters=2, n_init=10, random_state=42)
kmeans.fit(data)

plt.scatter(x, y, c=kmeans.labels_)
plt.title('K-means clustering (K=2)')
plt.show()

## Part 2: Detailed Program — Building K-Means From Scratch

To evaluate our own algorithm, we'll first generate a synthetic dataset of groups in 2-dimensional space using `make_blobs`, which creates groupings of 2D normal distributions and assigns a label corresponding to the group each point belongs to.

In [ ]:
import numpy as np
import random
import seaborn as sns
from numpy.random import uniform
from sklearn.datasets import make_blobs
from sklearn.preprocessing import StandardScaler

centers = 5

X_train, true_labels = make_blobs(n_samples=100, centers=centers, random_state=42)
X_train = StandardScaler().fit_transform(X_train)

sns.scatterplot(x=[X[0] for X in X_train],
                 y=[X[1] for X in X_train],
                 hue=true_labels,
                 palette="deep",
                 legend=None
                 )
plt.xlabel("x")
plt.ylabel("y")
plt.title("Ground-truth dataset for evaluating our K-means model")
plt.show()

This dataset provides a unique demonstration of the k-means algorithm: notice the orange point uncharacteristically far from its own cluster and sitting inside the purple cluster. This point can't be accurately classified as belonging to the right group, so even a well-working algorithm should incorrectly assign it to the purple group.

### Helper function: Euclidean distance

We'll need to calculate the distance between a point and a dataset of points multiple times, so we define a helper function.

In [ ]:
def euclidean(point, data):
    """
    Euclidean distance between point & data.
    Point has dimensions (m,), data has dimensions (n,m), and
    output will be of size (n,).
    """
    return np.sqrt(np.sum((point - data) ** 2, axis=1))

### A first (naive) implementation: random centroid initialization

First, the k-means clustering algorithm is initialized with a value for `k` and a maximum number of iterations for finding the optimal centroid locations (to avoid an infinite loop if it never converges).

Centroids start out randomly placed, uniformly distributed across the domain of the dataset. Then we iteratively:
1. Assign each point to its nearest centroid
2. Move each centroid to the mean of the points assigned to it
3. Repeat until the centroids stop moving (or we hit `max_iter`)

In [ ]:
class KMeansRandomInit:
    def __init__(self, n_clusters=8, max_iter=300):
        self.n_clusters = n_clusters
        self.max_iter = max_iter

    def fit(self, X_train):
        # Randomly select centroid start points, uniformly distributed across the domain of the dataset
        min_, max_ = np.min(X_train, axis=0), np.max(X_train, axis=0)
        self.centroids = [uniform(min_, max_) for _ in range(self.n_clusters)]

        # Iterate, adjusting centroids until converged or until passed max_iter
        iteration = 0
        prev_centroids = None
        while np.not_equal(self.centroids, prev_centroids).any() and iteration < self.max_iter:
            # Sort each datapoint, assigning to nearest centroid
            sorted_points = [[] for _ in range(self.n_clusters)]
            for x in X_train:
                dists = euclidean(x, self.centroids)
                centroid_idx = np.argmin(dists)
                sorted_points[centroid_idx].append(x)

            # Push current centroids to previous, reassign centroids as mean of the points belonging to them
            prev_centroids = self.centroids
            self.centroids = [np.mean(cluster, axis=0) if len(cluster) > 0 else prev_centroids[i]
                               for i, cluster in enumerate(sorted_points)]
            for i, centroid in enumerate(self.centroids):
                if np.isnan(centroid).any():  # Catch any np.nans, resulting from a centroid having no points
                    self.centroids[i] = prev_centroids[i]
            iteration += 1

    def evaluate(self, X):
        centroids = []
        centroid_idxs = []
        for x in X:
            dists = euclidean(x, self.centroids)
            centroid_idx = np.argmin(dists)
            centroids.append(self.centroids[centroid_idx])
            centroid_idxs.append(centroid_idx)
        return centroids, centroid_idxs

### First model evaluation

Let's train and test the naive (random-init) model on our dataset. Points are colored by true label and marker-styled by predicted cluster; `+` marks show the final centroid locations.

Try re-running this cell a few times — with purely random initialization you'll often see failure modes: a centroid stranded far from any group with no points, or two centroids splitting what should be a single cluster.

In [ ]:
kmeans_naive = KMeansRandomInit(n_clusters=centers)
kmeans_naive.fit(X_train)

# View results
class_centers, classification = kmeans_naive.evaluate(X_train)
sns.scatterplot(x=[X[0] for X in X_train],
                 y=[X[1] for X in X_train],
                 hue=true_labels,
                 style=classification,
                 palette="deep",
                 legend=None
                 )
plt.plot([c[0] for c in kmeans_naive.centroids],
         [c[1] for c in kmeans_naive.centroids],
         '+',
         markersize=10,
         )
plt.title("Random initialization — results vary run to run")
plt.show()

## Part 3: Re-evaluating Centroid Initialization

The naive model doesn't perform reliably. Two primary problems show up:

1. If a centroid is initialized far from any groups, it's unlikely to ever move.
2. If centroids are initialized too close together, they're unlikely to diverge from one another.

We fix this with a smarter initialization strategy: **k-means++**.

1. Initialize the first centroid as a random selection of one of the data points.
2. Calculate the sum of the distances between each data point and all current centroids.
3. Select the next centroid randomly, with probability proportional to the total distance to the centroids (so points far from existing centroids are more likely to be picked).
4. Repeat step 2–3 until all centroids have been initialized.

In [ ]:
class KMeans:
    def __init__(self, n_clusters=8, max_iter=300):
        self.n_clusters = n_clusters
        self.max_iter = max_iter

    def fit(self, X_train):
        # Initialize the centroids using the "k-means++" method: a random datapoint is selected as the first,
        # then the rest are initialized with probabilities proportional to their distances to existing centroids.
        self.centroids = [random.choice(X_train)]

        for _ in range(self.n_clusters - 1):
            # Calculate distances from points to the centroids
            dists = np.sum([euclidean(centroid, X_train) for centroid in self.centroids], axis=0)
            # Normalize the distances
            dists /= np.sum(dists)
            # Choose remaining points based on their distances
            new_centroid_idx, = np.random.choice(range(len(X_train)), size=1, p=dists)
            self.centroids += [X_train[new_centroid_idx]]

        # Iterate, adjusting centroids until converged or until passed max_iter
        iteration = 0
        prev_centroids = None
        while np.not_equal(self.centroids, prev_centroids).any() and iteration < self.max_iter:
            # Sort each datapoint, assigning to nearest centroid
            sorted_points = [[] for _ in range(self.n_clusters)]
            for x in X_train:
                dists = euclidean(x, self.centroids)
                centroid_idx = np.argmin(dists)
                sorted_points[centroid_idx].append(x)

            # Push current centroids to previous, reassign centroids as mean of the points belonging to them
            prev_centroids = self.centroids
            self.centroids = [np.mean(cluster, axis=0) if len(cluster) > 0 else prev_centroids[i]
                               for i, cluster in enumerate(sorted_points)]
            for i, centroid in enumerate(self.centroids):
                if np.isnan(centroid).any():  # Catch any np.nans, resulting from a centroid having no points
                    self.centroids[i] = prev_centroids[i]
            iteration += 1

    def evaluate(self, X):
        centroids = []
        centroid_idxs = []
        for x in X:
            dists = euclidean(x, self.centroids)
            centroid_idx = np.argmin(dists)
            centroids.append(self.centroids[centroid_idx])
            centroid_idxs.append(centroid_idx)
        return centroids, centroid_idxs

### Final evaluation with k-means++ initialization

Run this a few times — convergence should be much more consistent than the naive random-init version above.

In [ ]:
# Create a dataset of 2D distributions
centers = 5
X_train, true_labels = make_blobs(n_samples=100, centers=centers, random_state=42)
X_train = StandardScaler().fit_transform(X_train)

# Fit centroids to dataset
kmeans = KMeans(n_clusters=centers)
kmeans.fit(X_train)

# View results
class_centers, classification = kmeans.evaluate(X_train)
sns.scatterplot(x=[X[0] for X in X_train],
                 y=[X[1] for X in X_train],
                 hue=true_labels,
                 style=classification,
                 palette="deep",
                 legend=None
                 )
plt.plot([c[0] for c in kmeans.centroids],
         [c[1] for c in kmeans.centroids],
         'k+',
         markersize=10,
         )
plt.title("k-means++ initialization — much more reliable convergence")
plt.show()

## Summary

- **Elbow method**: run K-means across a range of K values and plot inertia; the "elbow" of the curve is a good estimate for K.
- **Naive K-means**: random centroid initialization can get stuck — stranded centroids or centroids too close together that never diverge.
- **k-means++**: initializes centroids one at a time, favoring points far from existing centroids, which gives far more consistent, correct convergence.